In [5]:
import re
import pandas as pd

In [11]:
# Load the data from the specified path
df_friends = pd.read_csv(r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\friends.csv")
df_friends_balanced = pd.read_csv(r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\friends_balanced.csv")
df_groups = pd.read_csv(r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\groups.csv")
df_test = pd.read_csv(r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\test.csv")

In [30]:
df_friends

,text,emotion,emotion_code,processed_text
0,also I was the point person on my company’s tr...,neutral,6,also I was the point person on my company’s tr...
1,You must’ve had your hands full.,neutral,6,You must have had your hands full .
2,That I did. That I did.,neutral,6,That I did . That I did .
3,So let’s talk a little bit about your duties.,neutral,6,So let us talk a little bit about your duties .
4,My duties? All right.,surprise,4,My duties ? All right .
...,...,...,...,...
12153,"Yeah, sweetie.",neutral,6,"Yeah , sweetie ."
12154,"I mean we’re not, we’re not gonna live togethe...",sadness,1,"I mean we are not , we are not gonna live toge..."
12155,What? Oh my God! I’m gonna miss you so much!,sadness,1,What ? Oh my God ! I am gonna miss you so much !
12156,I’m gonna miss you!,sadness,1,I am gonna miss you !


In [31]:
df_friends_balanced

,text,emotion,emotion_code,processed_text
0,"Oh, did you catch him?!",surprise,4,"Oh , did you catch him ? !"
1,Oh man. Please tell me one of 'em is Ma.,disgust,5,Oh man . Please tell me one of 'em is Ma .
2,"Look, look at your man, Ewing. Nice shot. You ...",disgust,5,"Look , look at your man , Ewing . Nice shot . ..."
3,"I’m sorry Chandler, y’know you are such a swee...",sadness,1,"I am sorry Chandler , you know you are such a ..."
4,You bug me.,disgust,5,You bug me .
...,...,...,...,...
10495,Wow. My brother never even told me when he los...,surprise,4,Wow . My brother never even told me when he lo...
10496,You guys got anything to eat? I just went down...,surprise,4,You guys got anything to eat ? I just went dow...
10497,"Hey-hey, now he’s showing us his poking device.",surprise,4,"Hey-hey , now he’s showing us his poking device ."
10498,"Oh, a 16-hour sit-in for Greenpeace.",neutral,6,"Oh , a 16-hour sit-in for Greenpeace ."


In [32]:
df_groups

,text,emotion,emotion_code,sub_emotion,processed_text
0,Did you kiss?,surprise,4,curiosity,Did you kiss ?
1,Yeah.,neutral,6,neutral,Yeah .
2,But no !,anger,2,annoyance,But no !
3,Why do you have a problem?,surprise,4,confusion,Why do you have a problem ?
4,Did you kiss?,surprise,4,curiosity,Did you kiss ?
...,...,...,...,...,...
24290,And tribes put everything on the line.,happiness,0,optimism,And tribes put everything on the line .
24291,I came in claiming that I had the most knowled...,happiness,0,pride,I came in claiming that I had the most knowled...
24292,At no point during my time here was that dispr...,happiness,0,pride,At no point during my time here was that dispr...
24293,There is nothing in the last few days of my ex...,happiness,0,pride,There is nothing in the last few days of my ex...


In [33]:
df_test

,text,emotion,emotion_code,sub-emotion,processed_text
0,Hang on to your seats cuz Asia's Next Top Mode...,happiness,0,excitement,Hang on to your seats cuz Asia's Next Top Mode...
1,Thousands of model hopefuls from all over Asia...,happiness,0,optimism,Thousands of model hopefuls from all over Asia...
2,But only the standout modeling talent were cho...,happiness,0,pride,But only the standout modeling talent were cho...
3,Prepare for an adventure of a lifetime,happiness,0,excitement,Prepare for an adventure of a lifetime
4,All I can say girls for this fierce fifth seas...,happiness,0,excitement,All I can say girls for this fierce fifth seas...
...,...,...,...,...,...
855,Won't give up. I'll stand up after this,happiness,0,optimism,Won't give up . I'll stand up after this
856,Next time on Asia's Next Top Model,neutral,6,neutral,Next time on Asia's Next Top Model
857,I'm very stressed very very stressed and the s...,fear,3,fear,I am very stressed very very stressed and the ...
858,"Yes, we're gossiping. She's beautiful, but she...",anger,2,disapproval,"Yes , we're gossiping . She's beautiful , but ..."


___

Before building the full pipeline, I'd recommend we start by:

Examining your datasets to understand their structure

Building a baseline model with GloVe + BiLSTM

Adding contextual features incrementally

Evaluating on validation set

Moving to more complex models like transformers

Would you like me to provide the code for any specific part of this plan first? Or would you like me to first examine your datasets in more detail to refine our approach?

In [34]:
def examine_datasets(dataframes):
    """Examine key characteristics of the datasets"""
    for name, df in dataframes.items():
        print(f"\nExamining dataset: {name}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        
        # Check for text column
        text_col = next((col for col in df.columns if 'text' in col.lower()), None)
        if text_col:
            print(f"\nText column found: {text_col}")
            print(f"Sample texts:\n{df[text_col].head(2).to_string()}")
            print(f"Average text length: {df[text_col].str.len().mean():.1f} characters")
            
        # Check for emotion column
        emotion_col = next((col for col in df.columns if 'emotion' in col.lower()), None)
        if emotion_col:
            print(f"\nEmotion distribution:")
            print(df[emotion_col].value_counts())

In [39]:
# Wrapping the DataFrames in a dictionary
dataframes = {
    'df_friends': df_friends,
    'df_friends_balanced': df_friends_balanced,
    'df_groups': df_groups,
    'df_test': df_test
}

# Call the function
examine_datasets(dataframes)



Examining dataset: df_friends
Shape: (12158, 4)
Columns: ['text', 'emotion', 'emotion_code', 'processed_text']

Text column found: text
Sample texts:
0    also I was the point person on my company’s tr...
1                     You must’ve had your hands full.
Average text length: 45.1 characters

Emotion distribution:
emotion
neutral      5661
happiness    2040
anger        1500
surprise     1313
sadness       949
disgust       351
fear          344
Name: count, dtype: int64

Examining dataset: df_friends_balanced
Shape: (10500, 4)
Columns: ['text', 'emotion', 'emotion_code', 'processed_text']

Text column found: text
Sample texts:
0                     Oh, did you catch him?!
1    Oh man. Please tell me one of 'em is Ma.
Average text length: 47.4 characters

Emotion distribution:
emotion
surprise     1500
disgust      1500
sadness      1500
anger        1500
happiness    1500
fear         1500
neutral      1500
Name: count, dtype: int64

Examining dataset: df_groups
Shape: (24295, 5)

___

Text Feature Extraction Plan for Emotion Classification


In [40]:
import pandas as pd
import numpy as np
import spacy
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from collections import Counter

# Download necessary NLTK resources
nltk.download('vader_lexicon')
nltk.download('punkt')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

1. Linguistic Feature Extraction


In [42]:
# Load SpaCy model for linguistic feature extraction
nlp = spacy.load("en_core_web_lg")

def extract_linguistic_features(texts):
    """Extract linguistic features from texts using SpaCy"""
    features = []
    
    for text in texts:
        doc = nlp(text)
        
        # Basic text statistics
        n_tokens = len(doc)
        n_sentences = len(list(doc.sents))
        avg_token_length = sum(len(token.text) for token in doc) / max(1, n_tokens)
        
        # POS tag distribution (normalized counts)
        pos_counts = Counter([token.pos_ for token in doc])
        total_tokens = max(1, sum(pos_counts.values()))
        
        # Get normalized counts for key POS categories
        noun_ratio = pos_counts.get('NOUN', 0) / total_tokens
        verb_ratio = pos_counts.get('VERB', 0) / total_tokens
        adj_ratio = pos_counts.get('ADJ', 0) / total_tokens
        adv_ratio = pos_counts.get('ADV', 0) / total_tokens
        pron_ratio = pos_counts.get('PRON', 0) / total_tokens
        
        # Syntactic features
        n_roots = len([token for token in doc if token.dep_ == 'ROOT'])
        n_subjects = len([token for token in doc if 'subj' in token.dep_])
        n_objects = len([token for token in doc if 'obj' in token.dep_])
        
        # Punctuation features (often important for emotion)
        exclamation_count = text.count('!')
        question_count = text.count('?')
        
        # Negation features (important for emotional context)
        negation_count = len([token for token in doc if token.dep_ == 'neg'])
        
        # Create feature dictionary
        feature_dict = {
            'n_tokens': n_tokens,
            'n_sentences': n_sentences,
            'avg_token_length': avg_token_length,
            'noun_ratio': noun_ratio,
            'verb_ratio': verb_ratio,
            'adj_ratio': adj_ratio,
            'adv_ratio': adv_ratio,
            'pron_ratio': pron_ratio,
            'n_roots': n_roots,
            'n_subjects': n_subjects,
            'n_objects': n_objects,
            'exclamation_count': exclamation_count,
            'question_count': question_count,
            'negation_count': negation_count,
        }
        
        features.append(feature_dict)
    
    return pd.DataFrame(features)

2. Sentiment Analysis Features


In [44]:
def extract_sentiment_features(texts):
    """Extract sentiment features using VADER"""
    sid = SentimentIntensityAnalyzer()
    
    features = []
    for text in texts:
        sentiment_scores = sid.polarity_scores(text)
        features.append(sentiment_scores)
    
    return pd.DataFrame(features)

3. N-gram Features with TF-IDF


In [45]:
def extract_ngram_features(texts, max_features=1000):
    """Extract n-gram features using TF-IDF"""
    # Unigrams and bigrams with TF-IDF
    tfidf_vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=5,  # Minimum document frequency
        use_idf=True,
        sublinear_tf=True  # Apply sublinear tf scaling (1 + log(tf))
    )
    
    tfidf_features = tfidf_vectorizer.fit_transform(texts)
    feature_names = tfidf_vectorizer.get_feature_names_out()
    
    return tfidf_features, feature_names, tfidf_vectorizer

4. Emotion-Specific Features


In [46]:
def extract_emotion_lexicon_features(texts):
    """Extract features using emotion lexicons"""
    # Define emotion lexicons (simplified here - you might want to use more comprehensive ones)
    emotion_lexicons = {
        'happiness': ['happy', 'joy', 'glad', 'pleased', 'excited', 'love', 'smile', 'laugh'],
        'sadness': ['sad', 'sorrow', 'grief', 'unhappy', 'crying', 'tear', 'depressed'],
        'anger': ['angry', 'mad', 'furious', 'rage', 'annoyed', 'irritated', 'upset'],
        'fear': ['afraid', 'scared', 'frightened', 'terrified', 'worried', 'anxious'],
        'surprise': ['surprised', 'shocked', 'astonished', 'amazed', 'unexpected'],
        'disgust': ['disgusted', 'gross', 'revolting', 'nasty', 'yuck', 'ew']
    }
    
    features = []
    for text in texts:
        # Normalize text for lexicon matching
        text_lower = text.lower()
        words = set(re.findall(r'\b\w+\b', text_lower))
        
        # Count matches for each emotion
        emotion_scores = {}
        for emotion, lexicon in emotion_lexicons.items():
            count = sum(1 for word in lexicon if word in words)
            # Normalize by text length
            emotion_scores[f'{emotion}_lexicon_score'] = count / max(1, len(words))
        
        features.append(emotion_scores)
    
    return pd.DataFrame(features)

5. Syntactic Patterns Related to Emotions


In [47]:
def extract_syntactic_emotion_patterns(texts):
    """Extract syntactic patterns that often indicate emotions"""
    patterns = {
        'intensifiers': r'\b(really|very|so|extremely|absolutely|totally)\b',
        'emotional_amplifiers': r'\b(omg|wow|oh|ah|gosh|damn|god)\b',
        'conditionals': r'\b(if|unless|until|whether)\b',
        'personal_pronouns': r'\b(I|me|my|mine|myself)\b',
        'hedges': r'\b(maybe|perhaps|possibly|probably|almost)\b',
        'certainty': r'\b(certainly|definitely|absolutely|undoubtedly)\b',
    }
    
    features = []
    for text in texts:
        text_lower = text.lower()
        pattern_counts = {}
        
        for pattern_name, pattern in patterns.items():
            matches = len(re.findall(pattern, text_lower))
            pattern_counts[f'{pattern_name}_count'] = matches
            # Normalize by text length (in words)
            word_count = len(text.split())
            pattern_counts[f'{pattern_name}_ratio'] = matches / max(1, word_count)
        
        features.append(pattern_counts)
    
    return pd.DataFrame(features)

6. Feature Integration Function


In [48]:
def extract_all_features(texts, include_tfidf=True, max_tfidf_features=1000):
    """Extract all features and combine them into a single feature set"""
    print("Extracting linguistic features...")
    linguistic_features = extract_linguistic_features(texts)
    
    print("Extracting sentiment features...")
    sentiment_features = extract_sentiment_features(texts)
    
    print("Extracting emotion lexicon features...")
    emotion_lexicon_features = extract_emotion_lexicon_features(texts)
    
    print("Extracting syntactic emotion patterns...")
    syntactic_patterns = extract_syntactic_emotion_patterns(texts)
    
    # Combine all DataFrame features
    all_features_df = pd.concat([
        linguistic_features,
        sentiment_features,
        emotion_lexicon_features,
        syntactic_patterns
    ], axis=1)
    
    if include_tfidf:
        print("Extracting TF-IDF features...")
        tfidf_features, feature_names, tfidf_vectorizer = extract_ngram_features(texts, max_tfidf_features)
        # Convert sparse matrix to DataFrame for inspection
        tfidf_df = pd.DataFrame.sparse.from_spmatrix(
            tfidf_features,
            columns=feature_names
        )
        return all_features_df, tfidf_features, tfidf_vectorizer
    
    return all_features_df, None, None

7. Applying to Your Dataset


In [49]:
def process_dataset(df, text_column='processed_text'):
    """Process a dataset to extract features"""
    texts = df[text_column].tolist()
    
    # Extract features
    features_df, tfidf_features, tfidf_vectorizer = extract_all_features(texts)
    
    # Display feature information
    print(f"Extracted {features_df.shape[1]} non-TF-IDF features")
    if tfidf_features is not None:
        print(f"Extracted {tfidf_features.shape[1]} TF-IDF features")
    
    return features_df, tfidf_features, tfidf_vectorizer

In [53]:
import pandas as pd
import numpy as np
import spacy
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from collections import Counter
import pickle

In [55]:
# Download necessary NLTK resources
nltk.download('vader_lexicon')
nltk.download('punkt')

# Load SpaCy model for linguistic feature extraction
print("Loading SpaCy model...")
nlp = spacy.load("en_core_web_lg")

def extract_features_and_enhance_df(df, text_column='processed_text', output_dir=None):
    """
    Extract features from text and add them to the DataFrame.
    
    Args:
        df: Input DataFrame with text data
        text_column: Column containing preprocessed text
        output_dir: Directory to save outputs (default is current directory)
    
    Returns:
        Enhanced DataFrame with all features
    """
    if output_dir is None:
        output_dir = r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured"
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Processing DataFrame with {len(df)} rows...")
    texts = df[text_column].tolist()
    
    # 1. Linguistic Features
    print("Extracting linguistic features...")
    linguistic_features = extract_linguistic_features(texts)
    
    # 2. Sentiment Features
    print("Extracting sentiment features...")
    sentiment_features = extract_sentiment_features(texts)
    
    # 3. Emotion Lexicon Features
    print("Extracting emotion lexicon features...")
    emotion_lexicon_features = extract_emotion_lexicon_features(texts)
    
    # 4. Syntactic Pattern Features
    print("Extracting syntactic pattern features...")
    syntactic_pattern_features = extract_syntactic_pattern_features(texts)
    
    # 5. TF-IDF Features
    print("Extracting TF-IDF features...")
    tfidf_features, tfidf_feature_names, tfidf_vectorizer = extract_tfidf_features(texts)
    
    # Combine all DataFrame features (excluding TF-IDF for now)
    print("Combining features...")
    feature_df = pd.concat([
        linguistic_features,
        sentiment_features,
        emotion_lexicon_features,
        syntactic_pattern_features
    ], axis=1)
    
    # Save TF-IDF vectorizer for future use
    tfidf_vectorizer_path = os.path.join(output_dir, "tfidf_vectorizer.pkl")
    with open(tfidf_vectorizer_path, 'wb') as f:
        pickle.dump(tfidf_vectorizer, f)
    print(f"TF-IDF vectorizer saved to {tfidf_vectorizer_path}")
    
    # Save TF-IDF features as a sparse matrix
    tfidf_features_path = os.path.join(output_dir, "tfidf_features.npz")
    from scipy.sparse import save_npz
    save_npz(tfidf_features_path, tfidf_features)
    print(f"TF-IDF features saved to {tfidf_features_path}")
    
    # Add all extracted features to the original DataFrame
    enhanced_df = df.copy()
    for col in feature_df.columns:
        enhanced_df[col] = feature_df[col]
    
    # Save the enhanced DataFrame
    enhanced_df_path = os.path.join(output_dir, "enhanced_dataset.csv")
    enhanced_df.to_csv(enhanced_df_path, index=False)
    print(f"Enhanced DataFrame saved to {enhanced_df_path}")
    
    # Save feature names for reference
    feature_names = list(feature_df.columns)
    feature_names_path = os.path.join(output_dir, "feature_names.txt")
    with open(feature_names_path, 'w') as f:
        for feature in feature_names:
            f.write(f"{feature}\n")
    print(f"Feature names saved to {feature_names_path}")
    
    # Save TF-IDF feature names
    tfidf_feature_names_path = os.path.join(output_dir, "tfidf_feature_names.txt")
    with open(tfidf_feature_names_path, 'w') as f:
        for feature in tfidf_feature_names:
            f.write(f"{feature}\n")
    print(f"TF-IDF feature names saved to {tfidf_feature_names_path}")
    
    return enhanced_df, tfidf_features, feature_names, tfidf_feature_names


def extract_linguistic_features(texts):
    """Extract linguistic features from texts using SpaCy"""
    features = []
    total_texts = len(texts)
    
    for i, text in enumerate(texts):
        if i % 1000 == 0:
            print(f"Processing linguistic features: {i}/{total_texts}")
            
        doc = nlp(text)
        
        # Basic text statistics
        n_tokens = len(doc)
        n_sentences = len(list(doc.sents))
        avg_token_length = sum(len(token.text) for token in doc) / max(1, n_tokens)
        
        # POS tag distribution (normalized counts)
        pos_counts = Counter([token.pos_ for token in doc])
        total_tokens = max(1, sum(pos_counts.values()))
        
        # Get normalized counts for key POS categories
        noun_ratio = pos_counts.get('NOUN', 0) / total_tokens
        verb_ratio = pos_counts.get('VERB', 0) / total_tokens
        adj_ratio = pos_counts.get('ADJ', 0) / total_tokens
        adv_ratio = pos_counts.get('ADV', 0) / total_tokens
        pron_ratio = pos_counts.get('PRON', 0) / total_tokens
        
        # Syntactic features
        n_roots = len([token for token in doc if token.dep_ == 'ROOT'])
        n_subjects = len([token for token in doc if 'subj' in token.dep_])
        n_objects = len([token for token in doc if 'obj' in token.dep_])
        
        # Punctuation features (often important for emotion)
        exclamation_count = text.count('!')
        question_count = text.count('?')
        
        # Capitalization features
        all_caps_count = sum(1 for token in doc if token.text.isupper() and len(token.text) > 1)
        
        # Negation features (important for emotional context)
        negation_count = len([token for token in doc if token.dep_ == 'neg'])
        
        # Create feature dictionary
        feature_dict = {
            'text_length': len(text),
            'n_tokens': n_tokens,
            'n_sentences': n_sentences,
            'avg_token_length': avg_token_length,
            'noun_ratio': noun_ratio,
            'verb_ratio': verb_ratio,
            'adj_ratio': adj_ratio,
            'adv_ratio': adv_ratio,
            'pron_ratio': pron_ratio,
            'n_roots': n_roots,
            'n_subjects': n_subjects,
            'n_objects': n_objects,
            'exclamation_count': exclamation_count,
            'question_count': question_count,
            'all_caps_count': all_caps_count,
            'negation_count': negation_count,
        }
        
        features.append(feature_dict)
    
    return pd.DataFrame(features)


def extract_sentiment_features(texts):
    """Extract sentiment features using VADER"""
    sid = SentimentIntensityAnalyzer()
    
    features = []
    total_texts = len(texts)
    
    for i, text in enumerate(texts):
        if i % 1000 == 0:
            print(f"Processing sentiment features: {i}/{total_texts}")
            
        sentiment_scores = sid.polarity_scores(text)
        features.append(sentiment_scores)
    
    return pd.DataFrame(features)


def extract_emotion_lexicon_features(texts):
    """Extract features using emotion lexicons"""
    # Define emotion lexicons for each category
    emotion_lexicons = {
        'happiness': ['happy', 'joy', 'glad', 'pleased', 'excited', 'love', 'smile', 'laugh', 'fun', 'wonderful', 'great', 'amazing'],
        'sadness': ['sad', 'sorrow', 'grief', 'unhappy', 'crying', 'tear', 'depressed', 'miserable', 'upset', 'hurt', 'disappointed'],
        'anger': ['angry', 'mad', 'furious', 'rage', 'annoyed', 'irritated', 'upset', 'hate', 'frustrated', 'offended'],
        'fear': ['afraid', 'scared', 'frightened', 'terrified', 'worried', 'anxious', 'nervous', 'panic', 'terror', 'dread'],
        'surprise': ['surprised', 'shocked', 'astonished', 'amazed', 'unexpected', 'wow', 'sudden', 'incredible', 'startled'],
        'disgust': ['disgusted', 'gross', 'revolting', 'nasty', 'yuck', 'ew', 'unpleasant', 'awful', 'terrible', 'horrible']
    }
    
    features = []
    total_texts = len(texts)
    
    for i, text in enumerate(texts):
        if i % 1000 == 0:
            print(f"Processing emotion lexicon features: {i}/{total_texts}")
            
        # Normalize text for lexicon matching
        text_lower = text.lower()
        words = set(re.findall(r'\b\w+\b', text_lower))
        
        # Count matches for each emotion
        emotion_scores = {}
        for emotion, lexicon in emotion_lexicons.items():
            count = sum(1 for word in lexicon if word in words)
            # Normalize by text length
            emotion_scores[f'{emotion}_lexicon_score'] = count / max(1, len(words))
        
        features.append(emotion_scores)
    
    return pd.DataFrame(features)


def extract_syntactic_pattern_features(texts):
    """Extract syntactic patterns that often indicate emotions"""
    patterns = {
        'intensifiers': r'\b(really|very|so|extremely|absolutely|totally)\b',
        'emotional_amplifiers': r'\b(omg|wow|oh|ah|gosh|damn|god)\b',
        'conditionals': r'\b(if|unless|until|whether)\b',
        'personal_pronouns': r'\b(I|me|my|mine|myself)\b',
        'hedges': r'\b(maybe|perhaps|possibly|probably|almost)\b',
        'certainty': r'\b(certainly|definitely|absolutely|undoubtedly)\b',
        'emphasis': r'\b(must|need|should|have to)\b',
        'taboo': r'\b(damn|shit|fuck|hell|crap)\b',
    }
    
    features = []
    total_texts = len(texts)
    
    for i, text in enumerate(texts):
        if i % 1000 == 0:
            print(f"Processing syntactic pattern features: {i}/{total_texts}")
            
        text_lower = text.lower()
        pattern_counts = {}
        
        for pattern_name, pattern in patterns.items():
            matches = len(re.findall(pattern, text_lower))
            pattern_counts[f'{pattern_name}_count'] = matches
            # Normalize by text length (in words)
            word_count = len(text.split())
            pattern_counts[f'{pattern_name}_ratio'] = matches / max(1, word_count)
        
        # Additional patterns
        pattern_counts['ellipsis_count'] = text.count('...')
        pattern_counts['quotes_count'] = text.count('"') / 2 + text.count("'")
        
        features.append(pattern_counts)
    
    return pd.DataFrame(features)


def extract_tfidf_features(texts, max_features=2000):
    """Extract TF-IDF features from texts"""
    # Initialize TF-IDF vectorizer
    tfidf_vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=5,  # Minimum document frequency
        max_df=0.95,  # Maximum document frequency
        ngram_range=(1, 2),  # Unigrams and bigrams
        sublinear_tf=True  # Apply sublinear tf scaling
    )
    
    # Fit and transform the texts
    tfidf_features = tfidf_vectorizer.fit_transform(texts)
    feature_names = tfidf_vectorizer.get_feature_names_out()
    
    print(f"Extracted {tfidf_features.shape[1]} TF-IDF features")
    return tfidf_features, feature_names, tfidf_vectorizer


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading SpaCy model...


In [61]:
# Main execution function
def main():
    """Main function to process datasets and extract features"""
    print("Loading datasets...")
    # Update with your actual paths
    df_friends_balanced = pd.read_csv(r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\test.csv")
    
    # Set output directory to the specified empty folder
    output_dir = r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\test"
    
    print("\nProcessing Friends Balanced dataset...")
    enhanced_df, tfidf_features, feature_names, tfidf_feature_names = extract_features_and_enhance_df(
        df_friends_balanced, 
        text_column='processed_text', 
        output_dir=output_dir
    )
    
    print("\nFeature extraction complete!")
    print(f"Total base features extracted: {len(feature_names)}")
    print(f"Total TF-IDF features extracted: {len(tfidf_feature_names)}")
    print(f"Enhanced DataFrame shape: {enhanced_df.shape}")
    
    # Print sample of extracted features
    print("\nSample of extracted features:")
    feature_cols = [col for col in enhanced_df.columns if col not in ['text', 'emotion', 'emotion_code', 'processed_text']]
    print(enhanced_df[feature_cols].head(3))
    
    print(f"\nAll output files have been saved to: {output_dir}")


if __name__ == "__main__":
    main()

Loading datasets...

Processing Friends Balanced dataset...
Processing DataFrame with 860 rows...
Extracting linguistic features...
Processing linguistic features: 0/860
Extracting sentiment features...
Processing sentiment features: 0/860
Extracting emotion lexicon features...
Processing emotion lexicon features: 0/860
Extracting syntactic pattern features...
Processing syntactic pattern features: 0/860
Extracting TF-IDF features...
Extracted 283 TF-IDF features
Combining features...
TF-IDF vectorizer saved to C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\test\tfidf_vectorizer.pkl
TF-IDF features saved to C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\test\tfidf_features.npz
Enhanced DataFrame saved to C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\test\enhanced_dataset.csv
Feature names saved to C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\test\feature_names.txt
TF-I

___

## __Re-do Features into one dataframe__

In [64]:
import pandas as pd
import numpy as np
import spacy
import re
import os
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from textblob import TextBlob
from sklearn.preprocessing import OneHotEncoder

# Download NLTK resources
nltk.download('vader_lexicon')
nltk.download('punkt')

# Load spaCy model
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_lg")

def extract_features(input_path, output_path, text_column='processed_text'):
    """
    Extract features following teacher's instructions:
    1. Lowercase all tokens
    2. Remove punctuation
    3. Perform sentiment analysis with VADER and TextBlob
    4. Extract POS tags and vectorize them
    
    Note: 
    - One-hot encoded emotion features are kept if they exist
    - Bag of words features are NOT included
    """
    print(f"Loading dataset from {input_path}...")
    df = pd.read_csv(input_path)
    
    print(f"Dataset shape: {df.shape}")
    print(f"Text column: {text_column}")
    
    # Create feature DataFrame
    featured_df = df.copy()
    
    # Get the texts to analyze
    texts = df[text_column].tolist()
    total_texts = len(texts)
    
    # 1. Lowercase all tokens and remove punctuation
    print("Preprocessing texts...")
    processed_texts = []
    for i, text in enumerate(texts):
        if i % 500 == 0:
            print(f"  Processing text {i}/{total_texts}")
        
        # Lowercase the text
        text = text.lower()
        
        # Remove punctuation
        text = re.sub(r'[^\w\s]', '', text)
        
        processed_texts.append(text)
    
    # Add processed text to DataFrame
    featured_df['cleaned_text'] = processed_texts
    
    # 2. Perform sentiment analysis with VADER
    print("Performing VADER sentiment analysis...")
    vader_features = get_vader_features(texts)
    
    # Add VADER features to DataFrame
    for col in vader_features.columns:
        featured_df[col] = vader_features[col]
    
    # 3. Perform sentiment analysis with TextBlob
    print("Performing TextBlob sentiment analysis...")
    textblob_features = get_textblob_features(texts)
    
    # Add TextBlob features to DataFrame
    for col in textblob_features.columns:
        featured_df[col] = textblob_features[col]
    
    # 4. Extract POS tags
    print("Extracting POS tags...")
    pos_features, pos_vectorized = get_pos_features(texts)
    
    # Add POS features to DataFrame
    featured_df['pos_tags'] = pos_features
    featured_df['pos_vectorized'] = pos_vectorized
    
    # Save featured dataset
    print(f"Saving featured dataset with {featured_df.shape[1]} columns to {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    featured_df.to_csv(output_path, index=False)
    
    # Generate feature summary
    generate_feature_summary(featured_df, output_path)
    
    print(f"Feature extraction complete. Total features: {featured_df.shape[1] - df.shape[1]}")
    
    # Show example of the first 5 rows with selected columns
    print("\nExample of featured data (first 5 rows):")
    display_columns = ['cleaned_text', 'vader_compound', 'textblob_polarity', 'textblob_subjectivity', 'pos_tags']
    if 'emotion' in featured_df.columns:
        display_columns = ['emotion'] + display_columns
    
    example_output = featured_df[display_columns].head(5)
    print(example_output)
    
    return featured_df

def get_vader_features(texts):
    """Extract sentiment features using VADER"""
    sid = SentimentIntensityAnalyzer()
    
    vader_neg = []
    vader_neu = []
    vader_pos = []
    vader_compound = []
    
    # Enhanced VADER features
    vader_intensity = []
    vader_sentiment_class = []
    
    total_texts = len(texts)
    
    for i, text in enumerate(texts):
        if i % 500 == 0:
            print(f"  VADER analysis: {i}/{total_texts}")
            
        # Get sentiment scores
        scores = sid.polarity_scores(text)
        
        vader_neg.append(scores['neg'])
        vader_neu.append(scores['neu'])
        vader_pos.append(scores['pos'])
        vader_compound.append(scores['compound'])
        
        # Calculate intensity (distance from neutral)
        intensity = abs(scores['compound'])
        vader_intensity.append(intensity)
        
        # Determine sentiment class
        if scores['compound'] >= 0.05:
            sentiment_class = 'positive'
        elif scores['compound'] <= -0.05:
            sentiment_class = 'negative'
        else:
            sentiment_class = 'neutral'
        vader_sentiment_class.append(sentiment_class)
    
    # Create DataFrame of VADER features
    vader_df = pd.DataFrame({
        'vader_neg': vader_neg,
        'vader_neu': vader_neu,
        'vader_pos': vader_pos,
        'vader_compound': vader_compound,
        'vader_intensity': vader_intensity,
        'vader_sentiment_class': vader_sentiment_class
    })
    
    return vader_df

def get_textblob_features(texts):
    """Extract sentiment features using TextBlob"""
    polarity = []
    subjectivity = []
    
    # Enhanced TextBlob features
    polarity_intensity = []
    subjectivity_class = []
    
    total_texts = len(texts)
    
    for i, text in enumerate(texts):
        if i % 500 == 0:
            print(f"  TextBlob analysis: {i}/{total_texts}")
            
        # Analyze text with TextBlob
        blob = TextBlob(text)
        
        # Get polarity and subjectivity
        polarity.append(blob.sentiment.polarity)
        subjectivity.append(blob.sentiment.subjectivity)
        
        # Calculate polarity intensity
        intensity = abs(blob.sentiment.polarity)
        polarity_intensity.append(intensity)
        
        # Determine subjectivity class
        if blob.sentiment.subjectivity < 0.3:
            subj_class = 'objective'
        elif blob.sentiment.subjectivity > 0.7:
            subj_class = 'subjective'
        else:
            subj_class = 'neutral'
        subjectivity_class.append(subj_class)
    
    # Create DataFrame of TextBlob features
    textblob_df = pd.DataFrame({
        'textblob_polarity': polarity,
        'textblob_subjectivity': subjectivity,
        'textblob_polarity_intensity': polarity_intensity,
        'textblob_subjectivity_class': subjectivity_class
    })
    
    return textblob_df

def get_pos_features(texts):
    """Extract POS tags and vectorize them"""
    pos_tags_list = []
    pos_vectorized_list = []
    
    # Create POS mapping dictionary
    pos_mapping = {}
    current_index = 0
    
    total_texts = len(texts)
    
    for i, text in enumerate(texts):
        if i % 500 == 0:
            print(f"  POS tagging: {i}/{total_texts}")
            
        doc = nlp(text)
        
        # Extract POS tags for this text
        pos_tags = [token.pos_ for token in doc]
        pos_tags_str = ", ".join(pos_tags)
        pos_tags_list.append(pos_tags_str)
        
        # Vectorize POS tags
        pos_vector = []
        for pos in pos_tags:
            if pos not in pos_mapping:
                pos_mapping[pos] = current_index
                current_index += 1
            pos_vector.append(str(pos_mapping[pos]))
        
        pos_vectorized_list.append(", ".join(pos_vector))
    
    return pos_tags_list, pos_vectorized_list

def generate_feature_summary(df, output_path):
    """Generate a summary of the features in the dataset"""
    # Create output directory
    output_dir = os.path.dirname(output_path)
    summary_path = os.path.join(output_dir, 'feature_summary.txt')
    
    # Identify feature categories
    original_cols = set(['text', 'processed_text', 'emotion', 'emotion_code', 
                      'sub_emotion', 'sub-emotion', 'emotion_joy', 'emotion_sadness', 
                      'emotion_anger', 'emotion_fear', 'emotion_surprise', 'emotion_disgust'])
    
    new_cols = [col for col in df.columns if col not in original_cols]
    vader_features = [col for col in new_cols if col.startswith('vader_')]
    textblob_features = [col for col in new_cols if col.startswith('textblob_')]
    pos_features = [col for col in new_cols if col.startswith('pos_') or col in ['pos_tags', 'pos_vectorized']]
    other_features = [col for col in new_cols if col not in vader_features + textblob_features + pos_features and col != 'cleaned_text']
    
    # Write summary
    with open(summary_path, 'w') as f:
        f.write("FEATURE SUMMARY\n")
        f.write("==============\n\n")
        
        f.write(f"Total features added: {len(new_cols)}\n\n")
        
        f.write("1. Preprocessing Features:\n")
        f.write("   - cleaned_text: Lowercase text with punctuation removed\n\n")
        
        f.write(f"2. VADER Sentiment Features ({len(vader_features)}):\n")
        for feature in vader_features:
            if df[feature].dtype == 'object':
                f.write(f"   - {feature}: Categorical with {df[feature].nunique()} unique values\n")
            else:
                f.write(f"   - {feature}: Numeric, range [{df[feature].min():.3f}, {df[feature].max():.3f}]\n")
        f.write("\n")
        
        f.write(f"3. TextBlob Features ({len(textblob_features)}):\n")
        for feature in textblob_features:
            if df[feature].dtype == 'object':
                f.write(f"   - {feature}: Categorical with {df[feature].nunique()} unique values\n")
            else:
                f.write(f"   - {feature}: Numeric, range [{df[feature].min():.3f}, {df[feature].max():.3f}]\n")
        f.write("\n")
        
        f.write(f"4. POS Tag Features ({len(pos_features)}):\n")
        for feature in pos_features:
            f.write(f"   - {feature}\n")
        f.write("\n")
        
        if other_features:
            f.write(f"5. Other Features ({len(other_features)}):\n")
            for feature in other_features:
                f.write(f"   - {feature}\n")
    
    print(f"Feature summary saved to {summary_path}")




[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading spaCy model...


In [65]:
# Usage
if __name__ == "__main__":
    # Process datasets
    input_datasets = [
        (r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\friends.csv",
         r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\friends_balanced\featured.csv"),

        (r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\friends_balanced.csv",
         r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\friends_balanced\featured_dataset.csv"),
        
        (r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\test.csv",
         r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\test\featured_dataset.csv"),
        
        (r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\groups.csv",
         r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\groups\featured_dataset.csv")
    ]
    
    for input_path, output_path in input_datasets:
        print(f"\nProcessing {os.path.basename(input_path)}...")
        featured_df = extract_features(input_path, output_path, text_column='processed_text')
        print(f"Completed processing {os.path.basename(input_path)}")


Processing friends.csv...
Loading dataset from C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\friends.csv...
Dataset shape: (12158, 4)
Text column: processed_text
Preprocessing texts...
  Processing text 0/12158
  Processing text 500/12158
  Processing text 1000/12158
  Processing text 1500/12158
  Processing text 2000/12158
  Processing text 2500/12158
  Processing text 3000/12158
  Processing text 3500/12158
  Processing text 4000/12158
  Processing text 4500/12158
  Processing text 5000/12158
  Processing text 5500/12158
  Processing text 6000/12158
  Processing text 6500/12158
  Processing text 7000/12158
  Processing text 7500/12158
  Processing text 8000/12158
  Processing text 8500/12158
  Processing text 9000/12158
  Processing text 9500/12158
  Processing text 10000/12158
  Processing text 10500/12158
  Processing text 11000/12158
  Processing text 11500/12158
  Processing text 12000/12158
Performing VADER sentiment analysis...
  VADER analysi

In [68]:
import pandas as pd
import numpy as np
import spacy
import re
import os
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from textblob import TextBlob
from sklearn.preprocessing import OneHotEncoder

# Download NLTK resources
nltk.download('vader_lexicon')
nltk.download('punkt')

# Load spaCy model
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_lg")

def extract_features(input_path, output_path, text_column='processed_text'):
    """
    Extract features following teacher's instructions:
    1. Lowercase all tokens
    2. Remove punctuation
    3. Perform sentiment analysis with VADER and TextBlob
    4. Extract POS tags and vectorize them
    
    Note: 
    - One-hot encoded emotion features are kept if they exist
    - Bag of words features are NOT included
    """
    print(f"Loading dataset from {input_path}...")
    df = pd.read_csv(input_path)
    
    print(f"Dataset shape: {df.shape}")
    print(f"Text column: {text_column}")
    
    # Create feature DataFrame
    featured_df = df.copy()
    
    # Get the texts to analyze
    texts = df[text_column].tolist()
    total_texts = len(texts)
    
    # 1. Lowercase all tokens and remove punctuation
    print("Preprocessing texts...")
    processed_texts = []
    for i, text in enumerate(texts):
        if i % 500 == 0:
            print(f"  Processing text {i}/{total_texts}")
        
        # Lowercase the text
        text = text.lower()
        
        # Remove punctuation
        text = re.sub(r'[^\w\s]', '', text)
        
        processed_texts.append(text)
    
    # Add processed text to DataFrame
    featured_df['cleaned_text'] = processed_texts
    
    # 2. Perform sentiment analysis with VADER
    print("Performing VADER sentiment analysis...")
    vader_features = get_vader_features(texts)
    
    # Add VADER features to DataFrame
    for col in vader_features.columns:
        featured_df[col] = vader_features[col]
    
    # 3. Perform sentiment analysis with TextBlob
    print("Performing TextBlob sentiment analysis...")
    textblob_features = get_textblob_features(texts)
    
    # Add TextBlob features to DataFrame
    for col in textblob_features.columns:
        featured_df[col] = textblob_features[col]
    
    # 4. Extract POS tags
    print("Extracting POS tags...")
    pos_features, pos_vectorized = get_pos_features(texts)
    
    # Add POS features to DataFrame
    featured_df['pos_tags'] = pos_features
    featured_df['pos_vectorized'] = pos_vectorized
    
    # Save featured dataset
    print(f"Saving featured dataset with {featured_df.shape[1]} columns to {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    featured_df.to_csv(output_path, index=False)
    
    # Generate feature summary
    generate_feature_summary(featured_df, output_path)
    
    print(f"Feature extraction complete. Total features: {featured_df.shape[1] - df.shape[1]}")
    
    # Show example of the first 5 rows with selected columns
    print("\nExample of featured data (first 5 rows):")
    display_columns = ['cleaned_text', 'vader_compound', 'textblob_polarity', 'textblob_subjectivity', 'pos_tags']
    if 'emotion' in featured_df.columns:
        display_columns = ['emotion'] + display_columns
    
    example_output = featured_df[display_columns].head(5)
    print(example_output)
    
    return featured_df

def get_vader_features(texts):
    """Extract sentiment features using VADER"""
    sid = SentimentIntensityAnalyzer()
    
    vader_neg = []
    vader_neu = []
    vader_pos = []
    vader_compound = []
    
    # Enhanced VADER features
    vader_intensity = []
    vader_sentiment_class = []
    
    total_texts = len(texts)
    
    for i, text in enumerate(texts):
        if i % 500 == 0:
            print(f"  VADER analysis: {i}/{total_texts}")
        
        # Get sentiment scores
        scores = sid.polarity_scores(text)
        
        vader_neg.append(scores['neg'])
        vader_neu.append(scores['neu'])
        vader_pos.append(scores['pos'])
        vader_compound.append(scores['compound'])
        
        # Calculate intensity (distance from neutral)
        intensity = abs(scores['compound'])
        vader_intensity.append(intensity)
        
        # Determine sentiment class
        if scores['compound'] >= 0.05:
            sentiment_class = 'positive'
        elif scores['compound'] <= -0.05:
            sentiment_class = 'negative'
        else:
            sentiment_class = 'neutral'
        vader_sentiment_class.append(sentiment_class)
    
    # Create DataFrame of VADER features
    vader_df = pd.DataFrame({
        'vader_neg': vader_neg,
        'vader_neu': vader_neu,
        'vader_pos': vader_pos,
        'vader_compound': vader_compound,
        'vader_intensity': vader_intensity,
        'vader_sentiment_class': vader_sentiment_class
    })
    
    return vader_df


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading spaCy model...


In [69]:
# Usage
if __name__ == "__main__":
    # Process datasets
    input_datasets = [
        (r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\friends.csv",
         r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\friends_balanced\featured.csv"),

        (r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\friends_balanced.csv",
         r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\friends_balanced\featured_dataset.csv"),
        
        (r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\test.csv",
         r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\test\featured_dataset.csv"),
        
        (r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\groups.csv",
         r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\groups\featured_dataset.csv")
    ]
    
    for input_path, output_path in input_datasets:
        print(f"\nProcessing {os.path.basename(input_path)}...")
        featured_df = extract_features(input_path, output_path, text_column='processed_text')
        print(f"Completed processing {os.path.basename(input_path)}")


Processing friends.csv...
Loading dataset from C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\friends.csv...
Dataset shape: (12158, 4)
Text column: processed_text
Preprocessing texts...
  Processing text 0/12158
  Processing text 500/12158
  Processing text 1000/12158
  Processing text 1500/12158
  Processing text 2000/12158
  Processing text 2500/12158
  Processing text 3000/12158
  Processing text 3500/12158
  Processing text 4000/12158
  Processing text 4500/12158
  Processing text 5000/12158
  Processing text 5500/12158
  Processing text 6000/12158
  Processing text 6500/12158
  Processing text 7000/12158
  Processing text 7500/12158
  Processing text 8000/12158
  Processing text 8500/12158
  Processing text 9000/12158
  Processing text 9500/12158
  Processing text 10000/12158
  Processing text 10500/12158
  Processing text 11000/12158
  Processing text 11500/12158
  Processing text 12000/12158
Performing VADER sentiment analysis...
  VADER analysi

In [70]:
df = pd.read_csv(r"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\friends_balanced\featured_dataset.csv")
df

,text,emotion,emotion_code,processed_text,cleaned_text,vader_neg,vader_neu,vader_pos,vader_compound,vader_intensity,vader_sentiment_class,textblob_polarity,textblob_subjectivity,textblob_polarity_intensity,textblob_subjectivity_class,pos_tags,pos_vectorized
0,"Oh, did you catch him?!",surprise,4,"Oh , did you catch him ? !",oh did you catch him,0.000,1.000,0.000,0.0000,0.0000,neutral,0.000000,0.000000,0.000000,objective,"INTJ, PUNCT, AUX, PRON, VERB, PRON, PUNCT, PUNCT","0, 1, 2, 3, 4, 3, 1, 1"
1,Oh man. Please tell me one of 'em is Ma.,disgust,5,Oh man . Please tell me one of 'em is Ma .,oh man please tell me one of em is ma,0.000,0.796,0.204,0.3182,0.3182,positive,0.000000,0.000000,0.000000,objective,"INTJ, NOUN, PUNCT, INTJ, VERB, PRON, NUM, ADP,...","0, 5, 1, 0, 4, 3, 6, 7, 3, 2, 8, 1"
2,"Look, look at your man, Ewing. Nice shot. You ...",disgust,5,"Look , look at your man , Ewing . Nice shot . ...",look look at your man ewing nice shot you ...,0.000,0.877,0.123,0.4215,0.4215,positive,0.600000,1.000000,0.600000,subjective,"VERB, PUNCT, VERB, ADP, PRON, NOUN, PUNCT, PRO...","4, 1, 4, 7, 3, 5, 1, 8, 1, 8, 5, 1, 3, 4, 3, 1..."
3,"I’m sorry Chandler, y’know you are such a swee...",sadness,1,"I am sorry Chandler , you know you are such a ...",i am sorry chandler you know you are such a s...,0.118,0.610,0.271,0.6433,0.6433,positive,-0.050000,0.716667,0.050000,subjective,"PRON, AUX, ADJ, PROPN, PUNCT, PRON, VERB, PRON...","3, 2, 12, 8, 1, 3, 4, 3, 2, 11, 11, 12, 5, 13,..."
4,You bug me.,disgust,5,You bug me .,you bug me,0.000,1.000,0.000,0.0000,0.0000,neutral,0.000000,0.000000,0.000000,objective,"PRON, VERB, PRON, PUNCT","3, 4, 3, 1"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10495,Wow. My brother never even told me when he los...,surprise,4,Wow . My brother never even told me when he lo...,wow my brother never even told me when he los...,0.143,0.621,0.236,0.3612,0.3612,positive,0.100000,1.000000,0.100000,subjective,"INTJ, PUNCT, PRON, NOUN, ADV, ADV, VERB, PRON,...","0, 1, 3, 5, 14, 14, 4, 3, 10, 3, 4, 3, 5, 1"
10496,You guys got anything to eat? I just went down...,surprise,4,You guys got anything to eat ? I just went dow...,you guys got anything to eat i just went down...,0.000,1.000,0.000,0.0000,0.0000,neutral,-0.293519,0.446296,0.293519,neutral,"PRON, NOUN, VERB, PRON, PART, VERB, PUNCT, PRO...","3, 5, 4, 3, 9, 4, 1, 3, 14, 4, 7, 7, 8, 7, 11,..."
10497,"Hey-hey, now he’s showing us his poking device.",surprise,4,"Hey-hey , now he’s showing us his poking device .",heyhey now hes showing us his poking device,0.000,1.000,0.000,0.0000,0.0000,neutral,0.000000,0.000000,0.000000,objective,"INTJ, INTJ, INTJ, PUNCT, ADV, PRON, AUX, VERB,...","0, 0, 0, 1, 14, 3, 2, 4, 3, 3, 4, 5, 1"
10498,"Oh, a 16-hour sit-in for Greenpeace.",neutral,6,"Oh , a 16-hour sit-in for Greenpeace .",oh a 16hour sitin for greenpeace,0.000,1.000,0.000,0.0000,0.0000,neutral,0.000000,0.000000,0.000000,objective,"INTJ, PUNCT, DET, NUM, PUNCT, NOUN, NOUN, PUNC...","0, 1, 11, 6, 1, 5, 5, 1, 5, 7, 8, 1"


In [3]:
def extract_features(input_path, output_path, text_column='processed_text'):
    """
    Extract features following teacher's instructions:
    1. Lowercase all tokens
    2. Remove punctuation
    3. Perform sentiment analysis with VADER and TextBlob
    4. Extract POS tags and vectorize them
    
    Note: 
    - One-hot encoded emotion features are kept if they exist
    - Bag of words features are NOT included
    """
    print(f"Loading dataset from {input_path}...")
    df = pd.read_csv(input_path)
    
    print(f"Dataset shape: {df.shape}")
    print(f"Text column: {text_column}")
    
    # Create feature DataFrame
    featured_df = df.copy()
    
    # Get the texts to analyze
    texts = df[text_column].tolist()
    total_texts = len(texts)
    
    # 1. Lowercase all tokens and remove punctuation with proper spacing
    print("Preprocessing texts...")
    processed_texts = []
    for i, text in enumerate(texts):
        if i % 500 == 0:
            print(f"  Processing text {i}/{total_texts}")
        
        # Process text using the fix_spaces function
        text = fix_spaces(text)
        processed_texts.append(text)
    
    # Add processed text to DataFrame
    featured_df['cleaned_text'] = processed_texts
    
    # 2. Perform sentiment analysis with VADER
    print("Performing VADER sentiment analysis...")
    vader_features = get_vader_features(texts)  # Using original texts for sentiment
    
    # Add VADER features to DataFrame
    for col in vader_features.columns:
        featured_df[col] = vader_features[col]
    
    # 3. Perform sentiment analysis with TextBlob
    print("Performing TextBlob sentiment analysis...")
    textblob_features = get_textblob_features(texts)  # Using original texts for sentiment
    
    # Add TextBlob features to DataFrame
    for col in textblob_features.columns:
        featured_df[col] = textblob_features[col]
    
    # 4. Extract POS tags using the CLEANED text (changed from original)
    print("Extracting POS tags from cleaned text...")
    pos_features, pos_vectorized = get_pos_features(processed_texts)  # Using processed_texts instead of texts
    
    # Add POS features to DataFrame
    featured_df['pos_tags'] = pos_features
    featured_df['pos_vectorized'] = pos_vectorized
    
    # Save featured dataset
    print(f"Saving featured dataset with {featured_df.shape[1]} columns to {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    featured_df.to_csv(output_path, index=False)
    
    # Generate feature summary
    generate_feature_summary(featured_df, output_path)
    
    print(f"Feature extraction complete. Total features: {featured_df.shape[1] - df.shape[1]}")
    
    # Show example of the first 5 rows with selected columns
    print("\nExample of featured data (first 5 rows):")
    display_columns = ['cleaned_text', 'vader_compound', 'textblob_polarity', 'textblob_subjectivity', 'pos_tags']
    if 'emotion' in featured_df.columns:
        display_columns = ['emotion'] + display_columns
    
    example_output = featured_df[display_columns].head(5)
    print(example_output)
    
    return featured_df

def fix_spaces(text):
    """
    Fix spacing issues in text by:
    1. Lowercasing the text
    2. Replacing punctuation with spaces
    3. Normalizing whitespace (replacing multiple spaces with single space)
    4. Removing leading and trailing whitespace
    
    Args:
        text: Input text string
        
    Returns:
        Text with properly fixed spacing
    """
    if not isinstance(text, str):
        return ""
    
    # Lowercase the text
    text = text.lower()
    
    # Replace punctuation with spaces
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # Normalize whitespace (replace multiple spaces with a single space)
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

Version 2, chosen code

In [1]:
import os
import re
import pandas as pd
import numpy as np
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
import nltk
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Ensure NLTK resources are downloaded
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('taggers/averaged_perceptron_tagger')
    nltk.data.find('vader_lexicon')
except LookupError:
    nltk.download('punkt')
    nltk.download('averaged_perceptron_tagger')
    nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [5]:
import os
import re
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob

# Ensure NLTK resources are downloaded
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('taggers/averaged_perceptron_tagger')
    nltk.data.find('vader_lexicon')
except LookupError:
    nltk.download('punkt')
    nltk.download('averaged_perceptron_tagger')
    nltk.download('vader_lexicon')

def fix_spaces(text):
    """Standardize text: lowercase, remove punctuation, normalize whitespace"""
    if not isinstance(text, str):
        return ""
    
    # Lowercase the text
    text = text.lower()
    
    # Replace punctuation with spaces
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # Normalize whitespace (replace multiple spaces with a single space)
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

def get_vader_features(texts):
    """Extract VADER sentiment features matching reference format"""
    sid = SentimentIntensityAnalyzer()
    
    vader_neg = []
    vader_neu = []
    vader_pos = []
    vader_compound = []
    
    print("Processing VADER sentiment...")
    for i, text in enumerate(texts):
        if i % 1000 == 0 and i > 0:
            print(f"Processed {i}/{len(texts)} texts")
            
        if isinstance(text, str):
            scores = sid.polarity_scores(text)
            vader_neg.append(scores['neg'])
            vader_neu.append(scores['neu'])
            vader_pos.append(scores['pos'])
            vader_compound.append(scores['compound'])
        else:
            vader_neg.append(0)
            vader_neu.append(0)
            vader_pos.append(0)
            vader_compound.append(0)
    
    # Create derived features
    vader_intensity = [abs(score) for score in vader_compound]
    vader_sentiment_class = [
        'positive' if score >= 0.05 else ('negative' if score <= -0.05 else 'neutral')
        for score in vader_compound
    ]
    
    # Create DataFrame
    features = pd.DataFrame({
        'vader_neg': vader_neg,
        'vader_neu': vader_neu,
        'vader_pos': vader_pos,
        'vader_compound': vader_compound,
        'vader_intensity': vader_intensity,
        'vader_sentiment_class': vader_sentiment_class
    })
    
    return features

def get_textblob_features(texts):
    """Extract TextBlob sentiment features matching reference format"""
    polarity = []
    subjectivity = []
    
    print("Processing TextBlob sentiment...")
    for i, text in enumerate(texts):
        if i % 1000 == 0 and i > 0:
            print(f"Processed {i}/{len(texts)} texts")
            
        if isinstance(text, str):
            blob = TextBlob(text)
            polarity.append(blob.sentiment.polarity)
            subjectivity.append(blob.sentiment.subjectivity)
        else:
            polarity.append(0)
            subjectivity.append(0)
    
    # Create derived features
    polarity_intensity = [abs(score) for score in polarity]
    subjectivity_class = [
        'objective' if score < 0.33 else ('subjective' if score > 0.66 else 'neutral')
        for score in subjectivity
    ]
    
    # Create DataFrame
    features = pd.DataFrame({
        'textblob_polarity': polarity,
        'textblob_subjectivity': subjectivity,
        'textblob_polarity_intensity': polarity_intensity,
        'textblob_subjectivity_class': subjectivity_class
    })
    
    return features

def get_universal_pos_tags(text):
    """Convert Penn Treebank tags to Universal POS tags for consistency with reference"""
    if not text:
        return [], []
        
    # Map from Penn Treebank to Universal POS tags
    tag_map = {
        'CC': 'CCONJ',   # coordinating conjunction
        'CD': 'NUM',     # cardinal digit
        'DT': 'DET',     # determiner
        'EX': 'PRON',    # existential there
        'FW': 'X',       # foreign word
        'IN': 'ADP',     # preposition/subordinating conjunction
        'JJ': 'ADJ',     # adjective
        'JJR': 'ADJ',    # adjective, comparative
        'JJS': 'ADJ',    # adjective, superlative
        'LS': 'X',       # list marker
        'MD': 'AUX',     # modal
        'NN': 'NOUN',    # noun, singular
        'NNS': 'NOUN',   # noun plural
        'NNP': 'PROPN',  # proper noun, singular
        'NNPS': 'PROPN', # proper noun, plural
        'PDT': 'DET',    # predeterminer
        'POS': 'PART',   # possessive ending
        'PRP': 'PRON',   # personal pronoun
        'PRP$': 'PRON',  # possessive pronoun
        'RB': 'ADV',     # adverb
        'RBR': 'ADV',    # adverb, comparative
        'RBS': 'ADV',    # adverb, superlative
        'RP': 'PART',    # particle
        'SYM': 'SYM',    # symbol
        'TO': 'PART',    # to
        'UH': 'INTJ',    # interjection
        'VB': 'VERB',    # verb, base form
        'VBD': 'VERB',   # verb, past tense
        'VBG': 'VERB',   # verb, gerund/present participle
        'VBN': 'VERB',   # verb, past participle
        'VBP': 'VERB',   # verb, sing. present, non-3d
        'VBZ': 'VERB',   # verb, 3rd person sing. present
        'WDT': 'DET',    # wh-determiner
        'WP': 'PRON',    # wh-pronoun
        'WP$': 'PRON',   # possessive wh-pronoun
        'WRB': 'SCONJ',  # wh-abverb
    }
    
    # Tokenize and get POS tags
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    
    # Convert to Universal tags
    universal_tags = [tag_map.get(tag, 'X') for _, tag in pos_tags]
    
    # Get unique tags while maintaining order
    unique_tags = []
    for tag in universal_tags:
        if tag not in unique_tags:
            unique_tags.append(tag)
    
    # Create indices
    indices = list(range(len(unique_tags)))
    
    return unique_tags, indices

def extract_features(input_path, output_path, text_column='processed_text'):
    """Extract features matching the reference dataframe format"""
    print(f"Loading dataset from {input_path}...")
    df = pd.read_csv(input_path)
    
    print(f"Dataset shape: {df.shape}")
    print(f"Text column: {text_column}")
    
    # Create feature DataFrame
    featured_df = df.copy()
    
    # Get the processed texts to analyze
    texts = df[text_column].tolist()
    total_texts = len(texts)
    
    # 1. Clean text
    print("Preprocessing texts...")
    processed_texts = [fix_spaces(text) for text in texts]
    featured_df['cleaned_text'] = processed_texts
    
    # 2. VADER Sentiment Analysis
    print("Extracting VADER features...")
    vader_features = get_vader_features(processed_texts)
    for col in vader_features.columns:
        featured_df[col] = vader_features[col]
    
    # 3. TextBlob Sentiment Analysis
    print("Extracting TextBlob features...")
    textblob_features = get_textblob_features(processed_texts)
    for col in textblob_features.columns:
        featured_df[col] = textblob_features[col]
    
    # 4. POS Tagging
    print("Extracting POS tag features...")
    pos_tags_list = []
    pos_vectorized_list = []
    
    for i, text in enumerate(processed_texts):
        if i % 1000 == 0 and i > 0:
            print(f"Processed POS tags for {i}/{total_texts} texts")
        
        unique_tags, indices = get_universal_pos_tags(text)
        pos_tags_list.append(", ".join(unique_tags))
        pos_vectorized_list.append(", ".join(str(idx) for idx in indices))
    
    featured_df['pos_tags'] = pos_tags_list
    featured_df['pos_vectorized'] = pos_vectorized_list
    
    # 5. Encode emotion labels
    if 'emotion' in featured_df.columns:
        print("Encoding emotion labels...")
        emotion_mapping = {
            'happiness': 0,
            'sadness': 1,
            'anger': 2,
            'fear': 3,
            'surprise': 4,
            'disgust': 5,
            'neutral': 6
        }
        
        featured_df['emotion'] = featured_df['emotion'].str.lower()
        featured_df['emotion_code'] = featured_df['emotion'].map(emotion_mapping)
    
    # Save featured dataset
    print(f"Saving featured dataset to {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    featured_df.to_csv(output_path, index=False)
    
    print(f"Feature extraction complete. Columns: {featured_df.shape[1]}")
    
    # Display sample
    display_columns = ['emotion', 'emotion_code', 'cleaned_text', 'vader_compound', 
                       'vader_intensity', 'vader_sentiment_class', 'textblob_polarity',
                       'textblob_subjectivity', 'pos_tags', 'pos_vectorized']
    sample = featured_df[display_columns].head(3)
    print("\nSample output:")
    print(sample)
    
    return featured_df



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ronle\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [6]:
data = "test_task7"

In [7]:
# Usage
if __name__ == "__main__":
    input_datasets = [
        (rf"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\{data}.csv",
         rf"C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\{data}.csv")
    ]
    
    for input_path, output_path in input_datasets:
        print(f"\nProcessing {os.path.basename(input_path)}...")
        featured_df = extract_features(input_path, output_path, text_column='processed_text')
        print(f"Completed processing {os.path.basename(input_path)}")


Processing test_task7.csv...
Loading dataset from C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\preprocessed\test_task7.csv...
Dataset shape: (860, 6)
Text column: processed_text
Preprocessing texts...
Extracting VADER features...
Processing VADER sentiment...
Extracting TextBlob features...
Processing TextBlob sentiment...
Extracting POS tag features...
Encoding emotion labels...
Saving featured dataset to C:\Users\ronle\Desktop\BUAS\2024-25c-fai2-adsai-group-group21\data\featured\test_task7.csv...
Feature extraction complete. Columns: 19

Sample output:
     emotion  emotion_code                                       cleaned_text  \
0  happiness             0  hang on to your seats because asia s next top ...   
1  happiness             0  thousands of model hopefuls from all over asia...   
2  happiness             0  but only the standout modeling talent were cho...   

   vader_compound  vader_intensity vader_sentiment_class  textblob_polarity  \
0          0

In [8]:
featured_df

,Unnamed: 0,text,emotion,sub-emotion,processed_text,emotion_code,cleaned_text,vader_neg,vader_neu,vader_pos,vader_compound,vader_intensity,vader_sentiment_class,textblob_polarity,textblob_subjectivity,textblob_polarity_intensity,textblob_subjectivity_class,pos_tags,pos_vectorized
0,0,Hang on to your seats cuz Asia's Next Top Mode...,happiness,excitement,Hang on to your seats because Asia's Next Top ...,0,hang on to your seats because asia s next top ...,0.000,0.878,0.122,0.2023,0.2023,positive,0.166667,0.166667,0.166667,objective,"NOUN, ADP, PART, PRON, ADJ, VERB, ADV, DET","0, 1, 2, 3, 4, 5, 6, 7"
1,1,Thousands of model hopefuls from all over Asia...,happiness,optimism,Thousands of model hopefuls from all over Asia...,0,thousands of model hopefuls from all over asia...,0.000,0.789,0.211,0.4588,0.4588,positive,0.000000,0.000000,0.000000,objective,"NOUN, ADP, DET, VERB, PART, PRON","0, 1, 2, 3, 4, 5"
2,2,But only the standout modeling talent were cho...,happiness,pride,But only the standout modeling talent were cho...,0,but only the standout modeling talent were cho...,0.000,0.748,0.252,0.5719,0.5719,positive,0.000000,1.000000,0.000000,subjective,"CCONJ, ADV, DET, NOUN, VERB, ADP, PRON, ADJ, NUM","0, 1, 2, 3, 4, 5, 6, 7, 8"
3,3,Prepare for an adventure of a lifetime,happiness,excitement,Prepare for an adventure of a lifetime,0,prepare for an adventure of a lifetime,0.000,0.685,0.315,0.3182,0.3182,positive,0.000000,0.000000,0.000000,objective,"NOUN, ADP, DET","0, 1, 2"
4,4,All I can say girls for this fierce fifth seas...,fear,excitement,All I can say girls for this fierce fifth seas...,3,all i can say girls for this fierce fifth seas...,0.000,0.888,0.112,0.2263,0.2263,positive,0.100000,1.000000,0.100000,subjective,"DET, NOUN, AUX, VERB, ADP, ADJ, PART","0, 1, 2, 3, 4, 5, 6"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
855,855,Won't give up. I'll stand up after this,neutral,optimism,Will not give up . I will stand up after this,6,will not give up i will stand up after this,0.000,1.000,0.000,0.0000,0.0000,neutral,0.000000,0.000000,0.000000,objective,"AUX, ADV, VERB, PART, NOUN, ADP, DET","0, 1, 2, 3, 4, 5, 6"
856,856,Next time on Asia's Next Top Model,neutral,neutral,Next time on Asia's Next Top Model,6,next time on asia s next top model,0.000,0.769,0.231,0.2023,0.2023,positive,0.166667,0.166667,0.166667,objective,"ADJ, NOUN, ADP","0, 1, 2"
857,857,I'm very stressed very very stressed and the s...,neutral,fear,I am very stressed very very stressed and the ...,6,i am very stressed very very stressed and the ...,0.502,0.498,0.000,-0.9199,0.9199,negative,-0.080000,0.563333,0.080000,neutral,"NOUN, VERB, ADV, ADJ, CCONJ, DET, ADP, PRON","0, 1, 2, 3, 4, 5, 6, 7"
858,858,"Yes, we're gossiping. She's beautiful, but she...",neutral,disapproval,"Yes , we are gossiping . She is beautiful , bu...",6,yes we are gossiping she is beautiful but she ...,0.245,0.577,0.177,-0.3919,0.3919,negative,0.183333,0.766667,0.183333,subjective,"ADP, PRON, VERB, ADJ, CCONJ, NUM, NOUN, PART","0, 1, 2, 3, 4, 5, 6, 7"


Most Beneficial Features for Emotion Classification

After analyzing the features we've extracted for the emotion classification task, I need to determine which 3-7 features would be most beneficial for model training. This requires understanding what signals best correlate with emotional expression in text.

From what I can see, we have several feature categories: VADER sentiment (neg, neu, pos, compound, intensity, class), TextBlob sentiment (polarity, subjectivity, polarity_intensity, subjectivity_class), and POS tagging information. For emotion classification, certain features will likely provide stronger signals than others.

Most Beneficial Features (in order of importance)

vader_compound - This provides a normalized, weighted composite score from -1 (extremely negative) to +1 (extremely positive), capturing the overall sentiment intensity that strongly correlates with emotional states.

vader_intensity - The absolute value of the compound score indicates emotional intensity regardless of direction, which is valuable for distinguishing between mild and strong emotions.

textblob_subjectivity - This measures how subjective or opinion-based the text is (0.0 to 1.0), which is particularly important for emotion detection since emotional expression is typically subjective.

pos_vectorized - The vectorized POS tags capture syntactic patterns that can reveal emotional expression (e.g., more pronouns in personal emotional statements, more exclamations in surprise).

textblob_polarity - This provides a different sentiment analysis algorithm's perspective, which may catch nuances that VADER misses.

vader_neg and vader_pos - These specific sentiment components can help distinguish between different negative emotions (sadness vs. anger) and positive emotions (joy vs. surprise).

These features provide a balanced approach that incorporates overall sentiment, emotional intensity, subjectivity, and syntactic patterns - all critical aspects of emotional expression in text.

 The combination should give a model strong signals to differentiate between the various emotion classes in your dataset.

___